In [ ]:
# =====================================
# 1. Imports
# =====================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn

# =====================================
# 2. Load Dataset
# =====================================
# Download the CSV file manually from:
# https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators
# and place it in your working directory.

df = pd.read_csv("CDC_diabetes_health_indicators_BRFSS2015.csv")

print("Shape:", df.shape)
df.head()



In [ ]:
# =====================================
# 3. Inspect & Prepare Data
# =====================================
print(df.info())

# Usually, the target column is 'Diabetes_binary'
target_col = "Diabetes_binary"
y = df[target_col]
X = df.drop(columns=[target_col])

# Most columns are already numeric/binary
# We'll just standardize continuous columns like BMI, Age, Income, etc.

continuous_cols = ["BMI", "Age", "Income"]
scaler = StandardScaler()
X[continuous_cols] = scaler.fit_transform(X[continuous_cols])

# Train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


In [ ]:
# =====================================
# 4. Convert to PyTorch Tensors
# =====================================
def to_tensor_dataset(X, y):
    X_tensor = torch.tensor(X.values, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.long)
    return TensorDataset(X_tensor, y_tensor)

train_ds = to_tensor_dataset(X_train, y_train)
val_ds = to_tensor_dataset(X_val, y_val)
test_ds = to_tensor_dataset(X_test, y_test)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=128)
test_dl = DataLoader(test_ds, batch_size=128)


In [ ]:
# =====================================
# 5. Define Neural Network
# =====================================
class DiabetesModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim//2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

input_dim = X.shape[1]
model = DiabetesModel(input_dim)
print(model)


In [ ]:
# =====================================
# 6. Training Loop
# =====================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)
            y_true.extend(yb.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    return np.array(y_true), np.array(y_pred)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg_loss = total_loss / len(train_dl.dataset)

    y_true, y_pred = evaluate(val_dl)
    acc = (y_true == y_pred).mean()
    print(f"Epoch {epoch+1:02d}/{num_epochs} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")


In [ ]:
# =====================================
# 7. Evaluate on Test Set
# =====================================
y_true, y_pred = evaluate(test_dl)
print("\nTest Accuracy:", (y_true == y_pred).mean())
print("\nClassification Report:\n", classification_report(y_true, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))


In [ ]:
# =====================================
# 8. Save & Load Model
# =====================================
torch.save(model.state_dict(), "diabetes_model.pth")

# Later / for inference:
# model = DiabetesModel(input_dim)
# model.load_state_dict(torch.load("diabetes_model.pth"))
# model.eval()
